In [ ]:
import cleanup
import plotting
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sbs
import math as math
import os
from os import listdir
import tkinter
from tkinter import filedialog
import scipy as scp
import scipy.stats as scpst

pd.options.mode.use_inf_as_na = True

In [ ]:
tkinter.Tk().withdraw()
filepath = filedialog.askdirectory()
primary_df = cleanup.data_to_dataframe(filepath)

In [ ]:
primary_df = primary_df[primary_df['Frame']<600]
primary_df = cleanup.calculate_distance_from_fixed_point(primary_df)
primary_df = cleanup.categorize_values(primary_df, 'X', 14, 1.5)

In [ ]:
primary_df_interp = cleanup.interpolate_missing_values(primary_df)
primary_df_interp = cleanup.calculate_distance_from_fixed_point(primary_df_interp)

In [ ]:
# df_neighbour = cleanup.create_combined_real_simulated_df(df = primary_df_interp)

In [ ]:
plotting.plot_speed_kde_by_genotype(primary_df_interp, genotype_col='Genotype')

In [ ]:
# for cond in primary_df_interp['Condition'].unique():    
#     plotting.plot_trajectory_heatmaps(primary_df_interp, cond)

In [ ]:
# sub_df = primary_df_interp[primary_df_interp['Genotype'] == 'WT']
# sub_df = sub_df[sub_df['Concentration'] == '10-4']
# sub_df = sub_df[sub_df['Collective'] == 'Group']
# sub_df = sub_df[sub_df['Odour'] == 'PA']

In [ ]:
sbs.lineplot(primary_df_interp, x = 'Frame', y = 'Distance', hue = 'Condition')

In [ ]:
plotting.plot_target_proximity_by_frame_bins(primary_df_interp)

In [ ]:
for cond in primary_df_interp['Condition'].unique():    
    plotting.sholl_analysis_over_time(primary_df_interp[primary_df_interp['Condition'] == cond])

In [ ]:
sub_df_5HT1A = primary_df_interp[(primary_df_interp['Genotype'] == 'TNTix5HT1A') | (primary_df_interp['Genotype'] == 'TNTex5HT1A')]

In [ ]:
cumulative_df = plotting.analyze_and_plot_target_acquisition(sub_df_5HT1A, assign_max_if_unreached=True)

In [ ]:
params = plotting.fit_logistic_to_success(cumulative_df)

In [ ]:
params

In [ ]:
sub_df_5HT7 = primary_df_interp[(primary_df_interp['Genotype'] == 'TNTix5HT7') | (primary_df_interp['Genotype'] == 'TNTex5HT7')]

In [ ]:
for cond in sub_df_5HT1A['Condition'].unique():    
    plotting.radial_sholl_heatmap_per_bin_normalized(sub_df_5HT1A[sub_df_5HT1A['Condition'] == cond], max_radius=22)

In [ ]:
plotting.plot_preference_index_over_time(sub_df_5HT7)

In [ ]:
cumulative_df = plotting.analyze_and_plot_target_acquisition(sub_df_5HT7, assign_max_if_unreached=True)

In [ ]:
params = plotting.fit_logistic_to_success(cumulative_df)

In [ ]:
params

In [ ]:
plotting.plot_preference_index_boxplots(primary_df_interp)

In [ ]:
# for cond in primary_df_interp['Condition'].unique():
#     plotting.plot_trajectory_heatmaps(primary_df_interp, cond, frame_bin_size=100, grid_size=1)

In [ ]:
# #Defining subdataframes for analysis

# df_EA = primary_df_interp[primary_df_interp['Odour'] == 'EA']
# df_1P = primary_df_interp[primary_df_interp['Odour'] == '1P']
# df_PA = primary_df_interp[primary_df_interp['Odour'] == 'PA']

# df_crosses = primary_df_interp[(primary_df_interp['Genotype'] == 'TrhxKir') 
#                               | (primary_df_interp['Genotype'] == 'CSDxKir')
#                               | (primary_df_interp['Genotype'] == 'WTxCSD')
#                               | (primary_df_interp['Genotype'] == 'WTxKir')
#                               | (primary_df_interp['Genotype'] == 'WTxTrh')]

In [ ]:
df_crosses_trh = primary_df_interp[(primary_df_interp['Genotype'] == 'TrhxKir') 
                              
                              
                              | (primary_df_interp['Genotype'] == 'WTxKir')
                              | (primary_df_interp['Genotype'] == 'WTxTrh')]

In [ ]:
df_crosses_csd = primary_df_interp[
                              (primary_df_interp['Genotype'] == 'CSDxKir')
                              | (primary_df_interp['Genotype'] == 'WTxCSD')
                              
                              | (primary_df_interp['Genotype'] == 'WTxTrh')]

In [ ]:
plotting.plot_two_distance_by_condition_averages(df_crosses_trh[df_crosses_trh['Starvation'] == 'Fed'],
                                                  df_crosses_trh[df_crosses_trh['Starvation'] == '5h'],
                                                    titles=('TrhxKir and Controls - Fed', 
                                                            'TrhxKir and Controls - 5h Starved'))

In [ ]:
plotting.plot_two_distance_by_condition_averages(df_crosses_csd[df_crosses_csd['Starvation'] == 'Fed'],
                                                  df_crosses_csd[df_crosses_csd['Starvation'] == '5h'],
                                                    titles=('CSDxKir and Controls - Fed', 
                                                            'CSDxKir and Controls - 5h Starved'))

In [ ]:
df_1P = primary_df_interp[primary_df_interp['Odour'] == '1P']

In [ ]:
plotting.plot_zone_means_subplot(df_1P)       

In [ ]:
avg_dist_df = df_neighbour.groupby(['Condition', 'GroupType'])['AvgNeighborDist'].mean().reset_index()

In [ ]:
# Simulated: aggregate by SimGroupID and Condition
sim_summary = (
    df_neighbour[df_neighbour['GroupType'] == 'Simulated']
    .groupby(['Condition', 'SimGroupID'])['AvgNeighborDist']
    .mean()
    .reset_index()
)
sim_summary['GroupType'] = 'Simulated'

# Real: aggregate by Trial and Condition
real_summary = (
    df_neighbour[df_neighbour['GroupType'] == 'Real']
    .groupby(['Condition', 'Trial'])['AvgNeighborDist']
    .mean()
    .reset_index()
)
real_summary['GroupType'] = 'Real'
real_summary.rename(columns={'Trial': 'GroupID'}, inplace=True)

# Match column names
sim_summary.rename(columns={'SimGroupID': 'GroupID'}, inplace=True)

# Combine both into a new plotting-friendly DataFrame
summary_df = pd.concat([real_summary, sim_summary], ignore_index=True)


In [ ]:
plt.figure(figsize=(12, 6))
sbs.boxplot(
    data=summary_df,
    x='Condition',
    y='AvgNeighborDist',
    hue='GroupType',
    palette='pastel',
    showfliers=False
)
sbs.stripplot(
    data=summary_df,
    x='Condition',
    y='AvgNeighborDist',
    hue='GroupType',
    dodge=True,
    color='black',
    size=4,
    alpha=0.6,
    legend=False
)
plt.xticks(rotation=90)
plt.ylabel('Avg Neighbor Distance')
plt.title('Average Neighbor Distances per Trial/SimGroup (One Point per Group)')
plt.legend(title='GroupType', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
df_neighbour_wt = df_neighbour[(df_neighbour['Genotype'] == 'WT') 
                               & (df_neighbour['Concentration'] == '10-4')
                               & (df_neighbour['Starvation'] == 'Fed')
                               & (df_neighbour['Odour'] == 'EA')]

In [ ]:
# Simulated: aggregate by SimGroupID and Condition
sim_summary = (
    df_neighbour_wt[df_neighbour_wt['GroupType'] == 'Simulated']
    .groupby(['Condition', 'SimGroupID'])['AvgNeighborDist']
    .mean()
    .reset_index()
)
sim_summary['GroupType'] = 'Simulated'

# Real: aggregate by Trial and Condition
real_summary = (
    df_neighbour_wt[df_neighbour_wt['GroupType'] == 'Real']
    .groupby(['Condition', 'Trial'])['AvgNeighborDist']
    .mean()
    .reset_index()
)
real_summary['GroupType'] = 'Real'
real_summary.rename(columns={'Trial': 'GroupID'}, inplace=True)

# Match column names
sim_summary.rename(columns={'SimGroupID': 'GroupID'}, inplace=True)

# Combine both into a new plotting-friendly DataFrame
summary_df = pd.concat([real_summary, sim_summary], ignore_index=True)

plt.figure(figsize=(12, 6))
sbs.boxplot(
    data=summary_df,
    x='Condition',
    y='AvgNeighborDist',
    hue='GroupType',
    palette='pastel',
    showfliers=False
)
sbs.stripplot(
    data=summary_df,
    x='Condition',
    y='AvgNeighborDist',
    hue='GroupType',
    dodge=True,
    color='black',
    size=4,
    alpha=0.6,
    legend=False
)
plt.xticks(rotation=90)
plt.ylabel('Avg Neighbor Distance')
plt.title('Average Neighbor Distances per Trial/SimGroup')
plt.legend(title='GroupType', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()



In [ ]:
plotting.plot_binned_summary(df_neighbour_wt)

In [ ]:
df_wt_real = primary_df_interp[(primary_df_interp['Genotype'] == 'WT') 
                               & (primary_df_interp['Concentration'] == '10-4')
                               & (primary_df_interp['Starvation'] == 'Fed')
                               & (primary_df_interp['Odour'] == 'EA')]

In [ ]:
df_1P_sub = df_1P[(df_1P['Concentration'] == '10-2') & (df_1P['Genotype'] == 'WT')]
              

In [ ]:
df_PA_sub = df_PA[(df_PA['Concentration'] == '10-4') & (df_PA['Genotype'] == 'Trh')]

In [ ]:
plotting.plot_zone_1_over_time(df_PA_sub)

In [ ]:
df_EA_sub = primary_df_interp[(primary_df_interp['Concentration'] == '10-4') 
             & (primary_df_interp['Genotype'] == 'Trh')
             & (primary_df_interp['Odour'] == 'EA')
             & (primary_df_interp['Collective'] == 'Group')]

In [ ]:
plotting.plot_preference_index_boxplots(df_EA_sub)

In [ ]:
plotting.plot_zone_means_subplot(df_EA_sub)

In [ ]:
plotting.plot_zone_1_over_time(df_EA_sub) 

In [ ]:
# plotting.plot_zone_means_subplot(df_PA_sub)

In [ ]:
plotting.boxplot_speed_by_genotype(primary_df_interp)

In [ ]:
plotting.plot_speed_kde_by_starvation(primary_df_interp)

In [ ]:
df_PA_sub = df_PA[(df_PA['Genotype'] == 'Trh') & (df_PA['Concentration'] == '10-4')]

In [ ]:
# plotting.plot_zone_means_subplot(df_PA_sub)

In [ ]:
plotting.plot_preference_index_over_time(df_PA_sub) 

In [ ]:
df_1P_sub = df_1P[(df_1P['Genotype'] == 'Trh') & (df_1P['Concentration'] == '10-2')]

In [ ]:
plotting.plot_preference_index_over_time(df_1P_sub) 